# Training Data Preparation (Split → Augment)

**Purpose:** produce the canonical train / validation / test splits for the allergen task
and the augmented training set, and save them to `data/final/splits/`.

## Why split first?

The previous pipeline augmented the **whole** dataset and only then split, so augmented
copies of a product could appear in the validation or test sets — a form of data leakage
that inflates scores. Here the stratified split happens first; **all augmentation is applied
to the training split only**. Validation and test sets contain only original products.

## Setup & Imports

In [7]:
import os
import random
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from utils.data_utils import (
    get_data_directories,
    load_labeled_data,
    augment_dataframe,
    augment_targeted,
    create_counter_examples,
    clean_training_data,
)
from utils.text_processing import (
    clean_html,
    get_allergen_list,
    allergens_to_binary,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Imports complete.")

Imports complete.


## 1. Load Dataset

Loads **Notebook 06's** freshly-labeled output (`labeled_dataset_enhanced.csv`). That file
is derived from the *current* rule engine, which counts **"may contain X"** facility
warnings as a positive (recall-first: a cross-contamination risk is still a risk). Loading
the stale July `_fixed` file would evaluate new rule semantics against old labels and
produce the 7 spurious tree-nut false positives seen in earlier runs.

In [8]:
dirs = get_data_directories()

# Canonical label source = Notebook 06's output, which re-derives `detected_allergens`
# with the CURRENT rule engine. This matters: the rule engine treats "may contain nuts"
# as a positive (recall-first / cross-contamination safety), and Notebook 06 regenerates
# the labels accordingly. Loading the stale `_fixed` file here would train on the OLD
# semantics while evaluating with the NEW one → spurious false positives downstream.
LABEL_SOURCE = os.path.join(dirs["final"], "labeled_dataset_enhanced.csv")
if not os.path.exists(LABEL_SOURCE):
    LABEL_SOURCE = os.path.join(dirs["final"], "labeled_dataset_enhanced_fixed.csv")
    print("WARNING: labeled_dataset_enhanced.csv (Notebook 06 output) not found — "
          "falling back to labeled_dataset_enhanced_fixed.csv (stale semantics). "
          "Run Notebook 06 first to regenerate labels.")
df = load_labeled_data(filepath=LABEL_SOURCE)
print(f"Dataset shape: {df.shape}  (source: {os.path.basename(LABEL_SOURCE)})")

df["ingredients_cleaned"] = df["ingredients_text_en"].apply(clean_html)
df["labels"] = df["detected_allergens"].apply(allergens_to_binary)

label_array = np.array(df["labels"].tolist())
print(f"Label distribution (per class): {label_array.sum(axis=0)}")

Dataset shape: (1057, 11)  (source: labeled_dataset_enhanced.csv)
Label distribution (per class): [500 103  89  53 235 347  88  51]


## 2. Optional: Label-Quality Cleaning

`clean_training_data` removes non-food / empty entries and fixes the *wheatflor → wheat*
label bug. It changes the row count slightly, so it is toggled here. The pre-restructure
reports used the full 1057 rows without this cleaning.

In [9]:
APPLY_CLEANING = True   # set to False to keep all 1057 rows

if APPLY_CLEANING:
    n_before = len(df)
    df = clean_training_data(df, ALLERGENS=get_allergen_list())
    removed = n_before - len(df)
    print(f"clean_training_data removed {removed} rows ({n_before} → {len(df)}).")
else:
    print("Skipping label-quality cleaning (keeping all rows).")

# Label vector re-derived after any row removals
label_array = np.array(df["labels"].tolist())
print(f"Final dataset: {len(df)} rows")

clean_training_data removed 4 rows (1057 → 1053).
Final dataset: 1053 rows


## 3. Stratified Split (FIRST — no leakage)

Mirrors `create_stratified_splits` (same two-stage `MultilabelStratifiedShuffleSplit`,
same seed) but operates on DataFrame positions so the full product metadata stays aligned
with each split.

In [10]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

positions = np.arange(len(df))
y = np.array(df["labels"].tolist())

# Stage 1: train vs (val + test)
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_pos, temp_pos = next(msss1.split(positions, y))

# Stage 2: val vs test within the temp split (val is 0.15 / 0.30 = 0.5 of temp)
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_local, test_local = next(msss2.split(temp_pos, y[temp_pos]))
val_pos, test_pos = temp_pos[val_local], temp_pos[test_local]

train_df = df.iloc[train_pos].copy().reset_index(drop=True)
val_df = df.iloc[val_pos].copy().reset_index(drop=True)
test_df = df.iloc[test_pos].copy().reset_index(drop=True)

# Convenience 'text' column everywhere downstream
for d in (train_df, val_df, test_df):
    d["text"] = d["ingredients_cleaned"]

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}  (total {len(df)})")

Train: 737  Val: 158  Test: 158  (total 1053)


## 4. Class Balance Per Split

The rare classes (tree_nuts has only ~40 positives total) are the reason for targeted
augmentation below. Check how the stratification distributed them.

In [11]:
allergen_list = get_allergen_list()
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    arr = np.array(d["labels"].tolist())
    print(f"\n{name} set per-class counts:")
    for i, a in enumerate(allergen_list):
        print(f"  {a:12s}: {int(arr[:, i].sum())}")


Train set per-class counts:
  milk        : 348
  eggs        : 72
  peanuts     : 62
  tree_nuts   : 37
  soy         : 164
  wheat       : 244
  fish        : 62
  shellfish   : 36

Val set per-class counts:
  milk        : 74
  eggs        : 15
  peanuts     : 14
  tree_nuts   : 8
  soy         : 36
  wheat       : 52
  fish        : 13
  shellfish   : 7

Test set per-class counts:
  milk        : 75
  eggs        : 16
  peanuts     : 13
  tree_nuts   : 8
  soy         : 35
  wheat       : 52
  fish        : 13
  shellfish   : 8


## 5. Augment the TRAIN Split Only

Three augmentation utilities with **different return semantics**:

| Function | Returns |
|----------|---------|
| `augment_dataframe` | **only new** synthetic rows |
| `augment_targeted`  | **originals + copies** (oversample rare classes) |
| `create_counter_examples` | **only new** counter-examples (teach the model that *coconut / shea / water chestnut* are NOT tree nuts) |

Because `augment_targeted` already includes the original rows, we concatenate
`[target_df, aug_df, counter_df]` — originals appear exactly once, and the synthetic
copies are added on top. Concatenating the original `train_df` *again* would double-count
every row, which was a bug in an earlier pipeline version.

In [12]:
# 5a. Generic augmentation — returns ONLY new synthetic rows
aug_df = augment_dataframe(train_df, num_augmented=2)
print(f"Generic augmentation added: {len(aug_df)} rows")

# 5b. Targeted augmentation for rare classes — returns ORIGINALS + copies
target_df = augment_targeted(
    train_df,
    target_allergens={"tree_nuts": 8, "shellfish": 3, "fish": 2, "peanuts": 2},
)
print(f"Targeted augmentation (originals + copies): {len(target_df)} rows")

# Source for counter-examples = original train rows (find trigger rows there)
train_aug_src = train_df

# 5c. Counter-examples for known tree-nuts false positives (new rows only)
# The last four blocks are the observed ML false positives from the held-out eval:
# plain green tea (ML fires tree_nuts @ 0.92) and blueberry-flavored desserts/drinks
# (fires @ 0.40). Teaching the model these do NOT imply tree nuts is exactly how the
# "difficult edge cases" in the user review get fixed without hurting recall.
tree_nuts_triggers = [
    "coconut", "coconut oil", "coconut milk", "coconut cream",
    "shea", "shea butter", "shea nut",
    "water chestnut", "water chestnuts",
    "bread", "noodles", "pasta", "breading",
    "bakery", "pastry", "corn nut", "chestnut honey",
    # observed ML FPs on the held-out test set
    "green tea", "green tea leaves", "green tea powder", "green tea extract", "matcha",
    "blueberry", "blueberry sauce", "blueberry flavor", "berry",
]
counter_df = create_counter_examples(
    train_aug_src,
    target_ingredients={"tree_nuts": tree_nuts_triggers},
    copies=3,
    random_state=SEED,
)
print(f"Counter-examples added: {len(counter_df)} rows")

# 5d. Combine — originals appear exactly once (via target_df), copies on top
train_augmented = pd.concat([target_df, aug_df, counter_df], ignore_index=True)
print(f"Final augmented train size: {len(train_augmented)} "
      f"(originals: {len(train_df)}, net added: {len(train_augmented) - len(train_df)})")

Generic augmentation added: 1232 rows
Targeted augmentation (originals + copies): 1389 rows
Counter-examples added: 762 rows
Final augmented train size: 3383 (originals: 737, net added: 2646)


## 6. Class Balance After Augmentation

In [13]:
aug_labels = np.array(train_augmented["labels"].tolist())
print(f"{'Allergen':<12} {'Train(orig)':>11} {'Train(aug)':>11}")
print("-" * 36)
for i, a in enumerate(allergen_list):
    print(f"{a:<12} {int(label_array[train_pos][:, i].sum()):>11} {int(aug_labels[:, i].sum()):>11}")

Allergen     Train(orig)  Train(aug)
------------------------------------
milk                 348        1704
eggs                  72         491
peanuts               62         569
tree_nuts             37         451
soy                  164         932
wheat                244        1394
fish                  62         478
shellfish             36         372


## 7. Save Canonical Split Files

Consumed by `08_allergen_training`, `09_allergen_evaluation`, and
`10_hybrid_evaluation`. The `labels` column is stored as a string list
(e.g. `"[1, 0, ...]"`) and parsed back with `ast.literal_eval` downstream.

In [14]:
split_dir = os.path.join(dirs["final"], "splits")
os.makedirs(split_dir, exist_ok=True)

def save_split(d, name):
    out = d.copy()
    out["labels"] = out["labels"].apply(lambda x: list(x))
    path = os.path.join(split_dir, name)
    out.to_csv(path, index=False)
    print(f"Saved {len(out)} rows → {path}")

save_split(train_df, "train.csv")           # raw original train (for per-seed ensemble augmentation)
save_split(train_augmented, "train_augmented.csv")
save_split(val_df, "val.csv")
save_split(test_df, "test.csv")

print("\nDone. Downstream notebooks load from data/final/splits/.")

Saved 737 rows → /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/final/splits/train.csv
Saved 3383 rows → /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/final/splits/train_augmented.csv
Saved 158 rows → /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/final/splits/val.csv
Saved 158 rows → /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/final/splits/test.csv

Done. Downstream notebooks load from data/final/splits/.


## Summary

- Split is stratified (70 / 15 / 15, seed 42) and happens **before** augmentation.
- Augmentation (generic + targeted + counter-examples) touched **only the train split**.
- `train.csv` = original train products; `train_augmented.csv` = train + synthetic copies.
- `val.csv` / `test.csv` contain only original products — used for selection / final eval.
- **Next:** run `08_allergen_training.ipynb`.